# 01 — Exploratory Data Analysis (EDA)

**Music Genre Classification from Spectrograms**

This notebook explores the GTZAN dataset, visualizes raw audio waveforms, mel-spectrograms,
and checks class balance before model training.

---
**Sections:**
1. Environment Setup & Config Loading
2. Dataset Overview & Class Distribution
3. Waveform Visualization
4. Mel-Spectrogram Visualization
5. Processed `.npy` Files Check
6. Observations & Next Steps

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display

# Add project root to path so we can import src modules
sys.path.insert(0, os.path.abspath('..'))

from src.utils import load_config, set_seed, plot_spectrogram

# Set random seed for reproducibility
set_seed(42)

# Load project configuration
config = load_config('../configs/config.yaml')
GENRES = config['data']['genres']
RAW_DIR = config['data']['raw_dir']
PROCESSED_DIR = config['data']['processed_dir']
SR = config['data']['sample_rate']
N_MELS = config['data']['n_mels']
HOP_LENGTH = config['data']['hop_length']
N_FFT = config['data']['n_fft']

print('Genres:', GENRES)
print('Sample rate:', SR, 'Hz')
print('n_mels:', N_MELS, '| hop_length:', HOP_LENGTH, '| n_fft:', N_FFT)

## 2. Dataset Overview & Class Distribution

In [ ]:
# Count audio files per genre in raw_dir
class_counts = {}
for genre in GENRES:
    genre_path = os.path.join(RAW_DIR, genre)
    if os.path.isdir(genre_path):
        files = [f for f in os.listdir(genre_path) if f.endswith('.wav')]
        class_counts[genre] = len(files)
    else:
        class_counts[genre] = 0

print('File counts per genre:')
for g, c in class_counts.items():
    print(f'  {g:12s}: {c} files')
print(f'Total: {sum(class_counts.values())} files')

In [ ]:
# Bar chart of class distribution
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(class_counts.keys(), class_counts.values(), color='steelblue', edgecolor='black')
ax.set_title('GTZAN Genre Distribution', fontsize=14)
ax.set_xlabel('Genre')
ax.set_ylabel('Number of Audio Files')
ax.set_ylim(0, max(class_counts.values()) + 10 if class_counts else 10)
plt.tight_layout()
plt.show()

## 3. Waveform Visualization

In [ ]:
# Pick the first available .wav file for each genre and plot its waveform
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
axes = axes.flatten()

for i, genre in enumerate(GENRES):
    genre_path = os.path.join(RAW_DIR, genre)
    if not os.path.isdir(genre_path):
        axes[i].set_title(f'{genre}\n(no data)')
        continue
    wav_files = sorted([f for f in os.listdir(genre_path) if f.endswith('.wav')])
    if not wav_files:
        axes[i].set_title(f'{genre}\n(no data)')
        continue
    fpath = os.path.join(genre_path, wav_files[0])
    y, sr = librosa.load(fpath, sr=SR, duration=5)  # first 5 seconds
    librosa.display.waveshow(y, sr=sr, ax=axes[i], color='navy', alpha=0.7)
    axes[i].set_title(genre.capitalize(), fontsize=10)
    axes[i].set_xlabel('')

plt.suptitle('Waveforms — First 5 Seconds per Genre', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Mel-Spectrogram Visualization

In [ ]:
# Plot mel-spectrograms for one example from each genre
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, genre in enumerate(GENRES):
    genre_path = os.path.join(RAW_DIR, genre)
    if not os.path.isdir(genre_path):
        axes[i].set_title(f'{genre}\n(no data)')
        continue
    wav_files = sorted([f for f in os.listdir(genre_path) if f.endswith('.wav')])
    if not wav_files:
        axes[i].set_title(f'{genre}\n(no data)')
        continue
    fpath = os.path.join(genre_path, wav_files[0])
    y, sr = librosa.load(fpath, sr=SR, duration=30)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, hop_length=HOP_LENGTH, n_fft=N_FFT)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel', ax=axes[i], cmap='magma')
    axes[i].set_title(genre.capitalize(), fontsize=10)

plt.suptitle('Mel-Spectrograms per Genre (dB scale)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Processed `.npy` Files Check

In [ ]:
# Check if processed .npy spectrograms exist
print('Checking processed spectrogram files...')
total_npy = 0
for genre in GENRES:
    npy_dir = os.path.join(PROCESSED_DIR, genre)
    count = 0
    if os.path.isdir(npy_dir):
        count = len([f for f in os.listdir(npy_dir) if f.endswith('.npy')])
    print(f'  {genre:12s}: {count} .npy files')
    total_npy += count

print(f'Total .npy files: {total_npy}')
if total_npy == 0:
    print('\n⚠️  No processed files found. Run the preprocessing script (Phase 2) first!')
else:
    # Load and inspect one sample
    sample_genre = GENRES[0]
    sample_file = os.path.join(PROCESSED_DIR, sample_genre, os.listdir(os.path.join(PROCESSED_DIR, sample_genre))[0])
    sample = np.load(sample_file)
    print(f'\nSample shape: {sample.shape}  (n_mels × time_frames)')
    print(f'Value range: min={sample.min():.2f}, max={sample.max():.2f}')

## 6. Observations & Next Steps

**Observations:**
- GTZAN contains 100 audio files per genre, 10 genres = **1,000 total samples** (30 seconds each at 22,050 Hz).
- Mel-spectrograms clearly show distinct frequency/energy patterns across genres (e.g., metal has high-energy, broadband noise; classical has structured harmonic patterns).
- Class distribution is perfectly balanced (100 files per genre) — no need for oversampling.

**Next Steps (Phase 2 — Preprocessing):**
- Convert all `.wav` files to mel-spectrograms and save as `.npy` arrays in `data/processed/`.
- Apply normalization (min-max or z-score) per spectrogram.
- Validate shapes and value ranges of all produced spectrograms.